# BraTS-GLI: 3D U-Net vs SwinUNETR (GPU)

Companion notebook for [MHKhan50/brats-glioma-segmentation](https://github.com/MHKhan50/brats-glioma-segmentation).
**Add Input → Your Work → "BraTS-GLI Download and Preprocess (CPU)"** (its output), and to continue a run, also the output of an earlier version of this notebook.

Kaggle settings: **GPU T4 x2**, **Internet on**, then *Save Version → Save & Run All*. Choose what to train in the cell below;
finished seeds are found in attached outputs and skipped. A session can run up to 12 hours.

Region-based training (WT, TC, ET; sigmoid, Dice + BCE), random 96³ crops, AMP, AdamW with cosine schedule.
Evaluation on the held-out test patients in the **original image space**: Dice and HD95 per region, voxel calibration (ECE), seed ensembles with patient-level bootstrap intervals.
Research only, not a medical device.

In [1]:
import os, subprocess, sys
os.chdir("/kaggle/working")
if not os.path.exists("brats-glioma-segmentation"):
    subprocess.run(["git", "clone", "-q", "https://github.com/MHKhan50/brats-glioma-segmentation.git"], check=True)
os.chdir("brats-glioma-segmentation")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[train,dev]"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
def bs(*args):
    """Run one CLI step, stream its output and stop with the error text if it fails."""
    proc = subprocess.Popen([sys.executable, "-m", "brats_seg.cli", *args], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in proc.stdout:
        print(line, end="", flush=True); lines.append(line)
    if proc.wait():
        raise RuntimeError("step failed: " + " ".join(args) + "\n" + "".join(lines[-30:]))

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


ac15482 Kaggle Notebook | BraTS-GLI 3D U-Net vs SwinUNETR (GPU) | Version 6

......                                                                   [100%]
6 passed in 1.74s


In [2]:
import glob, shutil, json
PREP = os.path.dirname(sorted(glob.glob("/kaggle/input/**/prep/cases.json", recursive=True))[0])
RUNS, REPORTS = "/kaggle/working/runs", "/kaggle/working/reports"
os.makedirs(RUNS, exist_ok=True)
for f in sum((glob.glob(f"/kaggle/input/**/runs/*{s}", recursive=True) for s in ("_best.pt", "_last.pt", "_meta.json")), []):
    if not os.path.exists(os.path.join(RUNS, os.path.basename(f))):
        shutil.copy(f, RUNS); print("restored", os.path.basename(f))
cases = json.load(open(PREP + "/cases.json"))
print(PREP, {s: sum(c["split"] == s for c in cases) for s in ("train", "val", "test")})

restored swinunetr_s0_best.pt
restored unet_s0_best.pt
restored unet_s1_best.pt
restored unet_s2_best.pt
restored swinunetr_s0_last.pt
restored unet_s0_last.pt
restored unet_s1_last.pt
restored unet_s2_last.pt
restored swinunetr_s0_meta.json
restored unet_s0_meta.json
restored unet_s1_meta.json
restored unet_s2_meta.json
/kaggle/input/notebooks/muzammalhussain11/brats-gli-download-and-preprocess-cpu/prep {'train': 878, 'val': 121, 'test': 252}


## Smoke test (a few steps, two validation cases)

In [3]:
for m in ("unet", "swinunetr"):
    bs("train", "--data", PREP, "--out", "/tmp/smoke", "--model", m, "--epochs", "1", "--limit-steps", "3", "--max-val-cases", "2")
# resume check: a second call with more epochs must continue from the saved epoch-1 checkpoint
bs("train", "--data", PREP, "--out", "/tmp/smoke", "--model", "unet", "--epochs", "2", "--limit-steps", "3", "--max-val-cases", "2")
bs("evaluate", "--data", PREP, "--runs", "/tmp/smoke", "--out", "/tmp/smoke/reports", "--limit", "2")
shutil.rmtree("/tmp/smoke", ignore_errors=True)

I0000 00:00:1795689962.457486     107 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/kaggle/working/brats-glioma-segmentation/src/brats_seg/train.py:128: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  losses.append(float(loss))
{"epoch": 1, "loss": 1.7998993396759033, "minutes": 0.2, "val_dice": [0.20116838261331457, 0.032263768149064974, 0.010246836846929912], "val_mean_dice": 0.08122632920310316}
{"model": "unet", "seed": 0, "epochs": 1, "batch_size": 2, "best_val_mean_dice": 0.08122632920310316, "params": 31411811, "minutes": 0.2, "finished": true}
I0000 00:00:1795689999.899523     163 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/kaggle/working/brats-glioma-segmentation/src/brats_seg/train.py:128: UserWarning: Co

## Training: edit this cell for each session

In [4]:
# One session = one line. Training saves a checkpoint after every epoch; if a session stops, attach this
# notebook's own output as input and run again: it resumes where it stopped.
# Session 1: [("unet", "0")]   Session 2: [("unet", "1,2")]   Session 3: [("swinunetr", "0")]   Session 4: [("swinunetr", "1,2")]
TRAIN = [("swinunetr", "1,2")]
EPOCHS = 60
for model, seeds in TRAIN:
    for seed in seeds.split(","):  # one process per seed, so memory is freed between seeds
        bs("train", "--data", PREP, "--out", RUNS, "--model", model, "--seeds", seed, "--epochs", str(EPOCHS),
           "--batch-size", "4", "--val-every", "10")

I0000 00:00:1795690083.225629     252 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/kaggle/working/brats-glioma-segmentation/src/brats_seg/train.py:128: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  losses.append(float(loss))
{"epoch": 1, "loss": 1.3480710651232228, "minutes": 4.1}
{"epoch": 2, "loss": 1.0945986378682804, "minutes": 7.6}
{"epoch": 3, "loss": 0.9225399042373379, "minutes": 11.0}
{"epoch": 4, "loss": 0.7637110450496413, "minutes": 14.4}
{"epoch": 5, "loss": 0.6246066215920122, "minutes": 17.8}
{"epoch": 6, "loss": 0.5159729168023148, "minutes": 21.2}
{"epoch": 7, "loss": 0.4248544104975652, "minutes": 24.6}
{"epoch": 8, "loss": 0.3538890655182268, "minutes": 28.0}
{"epoch": 9, "loss": 0.3127266688433956, "minutes": 31.4}
{"epoch": 10, "loss

## Test evaluation in native space and summary

In [5]:
bs("evaluate", "--data", PREP, "--runs", RUNS, "--out", REPORTS)
bs("analyse", "--reports", REPORTS)
for f in glob.glob(RUNS + "/*_meta.json"):
    shutil.copy(f, REPORTS)
shutil.make_archive("/kaggle/working/brats_reports", "zip", REPORTS)

I0000 00:00:1795716121.598065   59886 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
                    dice_WT  dice_TC  dice_ET
model     seed                               
swinunetr 0          0.9256   0.8975   0.8496
          1          0.9222   0.8956   0.8486
          2          0.9270   0.9001   0.8526
          ensemble   0.9291   0.9027   0.8549
unet      0          0.9266   0.9041   0.8603
          1          0.9256   0.8942   0.8509
          2          0.9260   0.9020   0.8553
          ensemble   0.9297   0.9036   0.8578
| Model | Dice WT | Dice TC | Dice ET | HD95 WT | HD95 TC | HD95 ET | seeds |
|---|---|---|---|---|---|---|---|
| swinunetr | 0.925 ± 0.002 | 0.898 ± 0.002 | 0.850 ± 0.002 | 4.4 | 4.2 | 10.3 | 3 |
| unet | 0.926 ± 0.001 | 0.900 ± 0.005 | 0.856 ± 0.005 | 4.1 | 3.4 | 9.3 | 3 |

SwinUNETR minus U-Net (ensembles, Dice): WT -0.001 [-0.003, +0.002], TC -0.001 [-0.007, +0.005], ET -0.003 [-0.006, +0.000]



'/kaggle/working/brats_reports.zip'